# 🧪 Pruebas Unitarias y CI/CD
## Algoritmos y Estructuras de Datos I — UADE

| Bloque | Tema |
|---|---|
| 1 | ¿Qué son las pruebas unitarias y por qué importan? |
| 2 | pytest — setup del entorno |
| 3 | Tests sobre `utils.py` — escribir y evolucionar los tests |
| 4 | Pruebas parametrizadas — `validar_email()` |
| 5 | Tests sobre `services.py` con mocking |
| 6 | CI/CD con GitHub Actions |

## 🛡️ Bloque 1 — ¿Qué son las pruebas unitarias?

Un programa que funciona hoy puede romperse mañana. Las pruebas unitarias son la **red de seguridad**: código que verifica otro código, de forma automática.

> 📺 Uncle Bob — *"Why I Stopped Reading Code and Building Harnesses"*: https://www.youtube.com/watch?v=5kCISBJwoZo
>
> *"If it's not tested, it's broken"* — no alcanza con leer el código, hay que probarlo automáticamente.

### El patrón AAA

| Fase | Qué hace | Ejemplo |
|---|---|---|
| **Arrange** | Preparar los datos de entrada | `id_usuario = "10"` |
| **Act** | Ejecutar la función | `resultado = validar_id_usuario(id_usuario)` |
| **Assert** | Verificar el resultado esperado | `assert resultado == True` |

### ¿Qué testear?

| ✅ Testear | ❌ No testear |
|---|---|
| Funciones de validación (`utils.py`) | El menú de `main.py` |
| Lógica de negocio (`services.py`) | El `input()` del usuario |
| Casos borde (string vacío, id=0) | Librerías externas (`re`, `json`) |

## 🔧 Bloque 2 — pytest: setup del entorno

Primero creamos un **entorno virtual** para aislar las dependencias del proyecto:

```bash
# 1. Crear el entorno virtual
python -m venv venv

# 2. Activarlo
source venv/bin/activate        # macOS / Linux
venv\Scripts\activate           # Windows

# 3. Instalar dependencias
pip install pytest tabulate

# 4. Verificar
python -m pytest --version
```

### `pytest.ini` mínimo

```ini
[pytest]
testpaths = test
pythonpath = src
```

### Convenciones de nombres

| Elemento | Convención | Ejemplo |
|---|---|---|
| Archivo | `test_<módulo>.py` | `test_utils.py` |
| Función | `test_<función>_<caso>` | `test_validar_id_usuario_valido` |

### Ejecutar el test
* Si usas el archivo de configuración **pytest.ini** directamente con `pytest` o `pytest -v`
* Si no hay pytest.ini entonces con `pytest -v test/nonmbre_archivo.py`

## 🔎 Bloque 3 — Tests sobre `utils.py`

`utils.py` contiene las validaciones. Son las más simples de testear: reciben una cadena y retornan `True` o `False`, sin dependencias externas.

La cátedra provee `test/test_utils.py` con un primer test:

```python
# test/test_utils.py
from src.utils import validar_id_usuario

def test_validar_id_usuario():
    # Arrange
    id_usuario = "10"
    # Act
    resultado = validar_id_usuario(id_usuario)
    # Assert
    assert resultado == True
```

Ese test verifica un caso válido. Pero un solo caso no es suficiente: **hay que cubrir también los casos que deberían fallar**.

### 3.1 — `test_validar_id_usuario`: agregando casos

El test inicial solo prueba `"10"`. Vamos a agregar más casos para cubrir **happy path** y **sad path**.

En el notebook simulamos la función que viene de `src/utils.py` para poder correr los tests directamente:

In [ ]:
import re

# Función tal como está en utils.py (patrón original)
def validar_id_usuario(id_usuario: str):
    patron = r"[0-9]+"
    return bool(re.fullmatch(patron, id_usuario))


# ── test/test_utils.py ────────────────────────────────────────────────────────
# Test provisto por la cátedra
def test_validar_id_usuario_valido():
    id_usuario = "10"
    resultado = validar_id_usuario(id_usuario)
    assert resultado == True

# Casos adicionales que escribimos nosotros
def test_validar_id_usuario_un_digito():
    assert validar_id_usuario("1") == True

def test_validar_id_usuario_cero_invalido():
    assert validar_id_usuario("0") == False   # 0 no es un id válido

def test_validar_id_usuario_negativo_invalido():
    assert validar_id_usuario("-3") == False

def test_validar_id_usuario_letras_invalido():
    assert validar_id_usuario("abc") == False

def test_validar_id_usuario_vacio_invalido():
    assert validar_id_usuario("") == False


# Corremos todos los tests
for fn in [
    test_validar_id_usuario_valido,
    test_validar_id_usuario_un_digito,
    test_validar_id_usuario_cero_invalido,
    test_validar_id_usuario_negativo_invalido,
    test_validar_id_usuario_letras_invalido,
    test_validar_id_usuario_vacio_invalido,
]:
    try:
        fn()
        print(f"✅ {fn.__name__}")
    except AssertionError as e:
        print(f"❌ FALLA  {fn.__name__}  →  {e}")

### El test descubrió un bug

`test_validar_id_usuario_cero_invalido` falla porque el patrón `r"[0-9]+"` acepta `"0"`: es un dígito válido en esa clase.

**El test no está mal — el patrón en `utils.py` está mal.**

Corrección en `src/utils.py`:

```python
# antes
patron = r"[0-9]+"

# después
patron = r"[1-9][0-9]*"   # primer dígito: 1–9  |  resto: 0–9 (opcional)
```

Rehacemos la corrida con el patrón corregido:

In [ ]:
# Función corregida en utils.py
def validar_id_usuario(id_usuario: str):
    patron = r"[1-9][0-9]*"   # corregido: excluye el 0
    return bool(re.fullmatch(patron, id_usuario))


# Los tests no cambian — solo cambia utils.py
for fn in [
    test_validar_id_usuario_valido,
    test_validar_id_usuario_un_digito,
    test_validar_id_usuario_cero_invalido,
    test_validar_id_usuario_negativo_invalido,
    test_validar_id_usuario_letras_invalido,
    test_validar_id_usuario_vacio_invalido,
]:
    try:
        fn()
        print(f"✅ {fn.__name__}")
    except AssertionError as e:
        print(f"❌ FALLA  {fn.__name__}  →  {e}")

### 3.2 — `test_validar_nombre`: el problema de los acentos

Aplicamos el mismo proceso: primero escribimos los tests con los casos que esperamos, y corremos para ver qué pasa.

In [ ]:
# Función tal como está en utils.py (patrón original)
def validar_nombre(nombre: str):
    patron = r"[a-zA-Z ]+"
    return bool(re.fullmatch(patron, nombre))


# ── test/test_utils.py ────────────────────────────────────────────────────────
def test_validar_nombre_simple():
    assert validar_nombre("Carlos") == True

def test_validar_nombre_con_acento():
    assert validar_nombre("María") == True      # acento — ¿qué pasa?

def test_validar_nombre_compuesto_con_acento():
    assert validar_nombre("Ana García") == True  # acento + espacio

def test_validar_nombre_numeros_invalido():
    assert validar_nombre("ana123") == False

def test_validar_nombre_simbolos_invalido():
    assert validar_nombre("Pedro!") == False

def test_validar_nombre_vacio_invalido():
    assert validar_nombre("") == False


for fn in [
    test_validar_nombre_simple,
    test_validar_nombre_con_acento,
    test_validar_nombre_compuesto_con_acento,
    test_validar_nombre_numeros_invalido,
    test_validar_nombre_simbolos_invalido,
    test_validar_nombre_vacio_invalido,
]:
    try:
        fn()
        print(f"✅ {fn.__name__}")
    except AssertionError as e:
        print(f"❌ FALLA  {fn.__name__}  →  {e}")

### El test descubrió otro bug

`[a-zA-Z ]` cubre solo el alfabeto inglés. Los acentos y la `ñ` quedan fuera.

Corrección en `src/utils.py`:

```python
# antes
patron = r"[a-zA-Z ]+"

# después
patron = r"[a-zA-ZáéíóúÁÉÍÓÚüÜñÑ ]+"
```

In [ ]:
# Función corregida en utils.py
def validar_nombre(nombre: str):
    patron = r"[a-zA-ZáéíóúÁÉÍÓÚüÜñÑ ]+"   # corregido: incluye acentos y ñ
    return bool(re.fullmatch(patron, nombre))


# Los tests no cambian
for fn in [
    test_validar_nombre_simple,
    test_validar_nombre_con_acento,
    test_validar_nombre_compuesto_con_acento,
    test_validar_nombre_numeros_invalido,
    test_validar_nombre_simbolos_invalido,
    test_validar_nombre_vacio_invalido,
]:
    try:
        fn()
        print(f"✅ {fn.__name__}")
    except AssertionError as e:
        print(f"❌ FALLA  {fn.__name__}  →  {e}")

## 📧 Bloque 4 — Pruebas parametrizadas: `validar_email()`

Si tenemos 8 casos para `validar_email`, escribir 8 funciones separadas es repetitivo y difícil de mantener.

`@pytest.mark.parametrize` permite escribir **un solo test con múltiples casos**.

El patrón actual es `r"[a-zA-Z]+@\w+[\.\w]+"`:
- `[a-zA-Z]+` → solo letras antes del `@` (no acepta dígitos ni puntos)
- `\w+` → dominio
- `[\.\w]+` → extensión (problema: acepta `.` al final sin nada después)

Veamos qué casos fallan:

In [ ]:
import re

def validar_email(email: str):
    patron = r"[a-zA-Z]+@\w+[\.\w]+"   # patrón original
    return bool(re.fullmatch(patron, email))

# Casos del @pytest.mark.parametrize — los corremos manualmente acá
casos_email = [
    # Happy path
    ("username@mail.com",          True),
    ("username.lastname@mail.com", True),
    ("username@mail.com.ar",       True),
    ("username1@mail.com",         True),   # tiene dígito — ¿falla?
    # Sad path
    ("username.com",               False),
    ("username@mail",              False),
    ("@mail.com",                  False),
    ("",                           False),
]

print(r"Con patrón original r'[a-zA-Z]+@\w+[\.\w]+' :")
for email, esperado in casos_email:
    resultado = validar_email(email)
    ok = "✅" if resultado == esperado else "❌ FALLA"
    print(f"  {ok}  '{email}' = {resultado}  (esperado: {esperado})")

### Ajuste del patrón de email

**Patrón corregido: `r"^[a-zA-Z0-9.]+@\w+(\.\w+)+$"`**

| Parte | Qué acepta |
|---|---|
| `^[a-zA-Z0-9.]+` | Usuario: letras, dígitos y punto |
| `@\w+` | Dominio principal (`mail`, `gmail`) |
| `(\.\w+)+` | Uno o más segmentos de extensión (`.com`, `.com.ar`) |
| `$` | Fin de cadena — no acepta caracteres extra al final |

La clave es `(\.\w+)+`: exige al menos un punto con algo después, por eso `"username@mail"` falla correctamente.

In [ ]:
def validar_email(email: str):
    patron = r"^[a-zA-Z0-9.]+@\w+(\.\w+)+$"
    return bool(re.fullmatch(patron, email))

print(r"Con patrón corregido r'^[a-zA-Z0-9.]+@\w+(\w+)+$' :")
for email, esperado in casos_email:
    resultado = validar_email(email)
    ok = "✅" if resultado == esperado else "❌ FALLA"
    print(f"  {ok}  '{email}' = {resultado}  (esperado: {esperado})")

print()
print("# test/test_utils.py — agregar esto")
code_str = (
    "import pytest\n"
    "from src.utils import validar_email\n\n"
    "@pytest.mark.parametrize('email, esperado', [\n"
    "    # Happy path\n"
    "    ('username@mail.com',          True),\n"
    "    ('username.lastname@mail.com', True),\n"
    "    ('username@mail.com.ar',       True),\n"
    "    ('username1@mail.com',         True),\n"
    "    # Sad path\n"
    "    ('username.com',               False),\n"
    "    ('username@mail',              False),\n"
    "    ('@mail.com',                  False),\n"
    "    ('',                           False),\n"
    "])\n"
    "def test_validar_email_parametrizado(email, esperado):\n"
    "    assert validar_email(email) == esperado\n\n"
    "# En lugar de 8 funciones, una sola con todos los casos \u2705"
)
print(code_str)

## 🎭 Bloque 5 — Tests sobre `services.py` con mocking

`services.py` depende de `storage.py` para leer y escribir en disco. En un test **no queremos tocar el disco**: sería lento y dejaría archivos basura.

El **mock** reemplaza `storage.read_data` y `storage.write_data` por versiones falsas que operan en memoria:

```
Test                   services.py             storage.py
  |—— crear_usuario ——►|                           |
  |                    |—— mock en lugar de ———————►| (no existe)
  |                    |◄— retorna lista fija ——————|
```

Vamos a crear el archivo `test/test_services.py`.

In [ ]:
from unittest.mock import patch

# Simulamos storage para poder correr el test en el notebook
class storage:
    @staticmethod
    def read_data(ruta): pass
    @staticmethod
    def write_data(ruta, datos): pass

from pathlib import Path
RUTA_USUARIOS = Path("data/usuarios.json")

def crear_usuario(nuevo_usuario):
    usuarios = storage.read_data(RUTA_USUARIOS)
    usuarios.append(nuevo_usuario)
    return storage.write_data(RUTA_USUARIOS, usuarios)

# ── test/test_services.py ─────────────────────────────────────────────────────
def test_crear_usuario():
    # Arrange
    usuarios = [
        {"id_usuario": 1, "nombre": "Elon", "email": "elon@tesla.com"},
        {"id_usuario": 2, "nombre": "Jeff", "email": "jeff@aws.com"},
        {"id_usuario": 3, "nombre": "Bill", "email": "Bill@microsoft.com"},
        {"id_usuario": 4, "nombre": "Mark", "email": "Mark@meta.com"},
    ]
    nuevo_usuario = {"id_usuario": 5, "nombre": "Lisa", "email": "Lisa@amd.com"}

    with patch.object(storage, "read_data", return_value=usuarios.copy()) as mock_read, \
         patch.object(storage, "write_data", return_value=True) as mock_write:

        # Act
        resultado = crear_usuario(nuevo_usuario)

        # Assert
        assert resultado is True
        lista_guardada = mock_write.call_args[0][1]
        assert len(lista_guardada) == 5
        assert lista_guardada[-1]["nombre"] == "Lisa"
        assert lista_guardada[-1]["id_usuario"] == 5

try:
    test_crear_usuario()
    print("✅ test_crear_usuario")
except AssertionError as e:
    print(f"❌ test_crear_usuario — {e}")

### El archivo real: `test/test_services.py`

Con los imports correctos del proyecto, el archivo quedaría así:

```python
# test/test_services.py
from unittest.mock import patch
from src import services

def test_crear_usuario():
    # Arrange
    usuarios = [
        {"id_usuario": 1, "nombre": "Elon", "email": "elon@tesla.com"},
        {"id_usuario": 2, "nombre": "Jeff", "email": "jeff@aws.com"},
        {"id_usuario": 3, "nombre": "Bill", "email": "Bill@microsoft.com"},
        {"id_usuario": 4, "nombre": "Mark", "email": "Mark@meta.com"},
    ]
    nuevo_usuario = {"id_usuario": 5, "nombre": "Lisa", "email": "Lisa@amd.com"}

    with patch("src.services.storage.read_data") as mock_read, \
         patch("src.services.storage.write_data") as mock_write:

        mock_read.return_value = usuarios.copy()
        mock_write.return_value = True

        # Act
        resultado = services.crear_usuario(nuevo_usuario)

        # Assert
        assert resultado is True
        mock_read.assert_called_once_with(services.RUTA_USUARIOS)
        mock_write.assert_called_once_with(
            services.RUTA_USUARIOS,
            usuarios + [nuevo_usuario]
        )
```

La diferencia clave: `patch("src.services.storage.read_data")` apunta al módulo donde la función **se usa**, no donde se define.

## 🚀 Bloque 6 — CI/CD con GitHub Actions

- **CI** (Continuous Integration): ejecuta los tests automáticamente en cada `git push`
- **Sin CI**: el bug llega a `main`. **Con CI**: el merge se bloquea hasta arreglar.

```
git push → GitHub Actions ejecuta pytest (~30s)
               ✅ todos pasan → merge habilitado
               ❌ alguno falla → merge bloqueado
```

El workflow vive en:
```
proyecto/
└── .github/
    └── workflows/
        └── ci.yml
```

In [ ]:
import os

os.makedirs(".github/workflows", exist_ok=True)

ci_yml = (
    "name: CI - Pruebas Unitarias\n\n"
    "on:\n"
    "  push:\n"
    "    branches: [ main, develop ]\n"
    "  pull_request:\n"
    "    branches: [ main ]\n\n"
    "jobs:\n"
    "  test:\n"
    "    runs-on: ubuntu-latest\n\n"
    "    steps:\n"
    "      - name: Checkout del codigo\n"
    "        uses: actions/checkout@v4\n\n"
    "      - name: Configurar Python 3.11\n"
    "        uses: actions/setup-python@v5\n"
    "        with:\n"
    "          python-version: \"3.11\"\n\n"
    "      - name: Instalar dependencias\n"
    "        run: |\n"
    "          pip install pytest tabulate\n\n"
    "      - name: Ejecutar tests\n"
    "        run: python -m pytest -v\n"
)

with open(".github/workflows/ci.yml", "w") as f:
    f.write(ci_yml)

print("✅ .github/workflows/ci.yml generado.")

### Subir a GitHub y activar el pipeline

```bash
# 1. Inicializar el repositorio (solo la primera vez)
git init
git remote add origin https://github.com/usuario/nombre-repo.git

# 2. Agregar el workflow de CI
git add .github/

# 3. Agregar los tests
git add test/

# 4. Commit
git commit -m "ci: agregar workflow y tests unitarios"

# 5. Push — esto dispara el pipeline automáticamente
git push -u origin main
```

Luego ir a GitHub → pestaña **Actions** para ver el pipeline en ejecución.

**Badge para el README.md:**
```markdown
![CI](https://github.com/usuario/nombre-repo/actions/workflows/ci.yml/badge.svg)
```

## 💭 Reflexión — Doble clic para editar

**1. ¿Por qué usamos mock en los tests de `services.py` en lugar de leer y escribir archivos reales?**
> _Tu respuesta acá_

**2. ¿Qué ventaja tiene `@pytest.mark.parametrize` sobre escribir una función de test por caso?**
> _Tu respuesta acá_

**3. Si `test_crear_usuario` falla después de que alguien modifica `generar_id_usuario()`, ¿qué información te da ese fallo y qué harías?**
> _Tu respuesta acá_